<a href="https://colab.research.google.com/github/maurocollin/bigdata-saude-niteroi/blob/main/T%C3%B3picos_de_Big_Data_em_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise de Indicadores de Saúde Municipal: Previsão de Demandas por Bairro em Niterói

#Estrutura do Projeto

1. Coleta de dados brutos de fontes oficiais em formatos como CSV.
- [x]

2. Limpeza e normalização dos dados com Pandas e PySpark.  

3. Tratar e verificar a veracidade e eliminar informações falsas ou irrelevantes.  

4. Aplcar cálculo de média, mediana e desvio padrão dos indicadores de saúde para entender a dispersão dos dados.  

5. Análise Estatística Descritiva:
     - Cálculo de Média, Mediana e Desvio Padrão dos indicadores de saúde para entender a dispersão dos dados.  

6. Distribuição de Frequências:
     - Organização das ocorrências de saúde por faixas (categorias) para identificar padrões de demanda.  

7. Identificação de Outliers:
     - Uso de Boxplots ou Z-score para detectar anomalias ou casos atípicos (como surtos repentinos em bairros específicos).  

8. Carga e Visualização (Load):
     - Apresentação dos resultados finais através de gráficos que demonstrem o impacto social gerado.

In [ ]:
!pip install geobr folium scipy  -q
import pandas as pd
import numpy as np
import geobr
import folium
from scipy import stats
import matplotlib.pyplot as plt

# caminho base (facilita manutenção depois)
caminho = '/content/drive/MyDrive/Colab Notebooks/DATA/'

print("Carregando arquivos...")

# leitura das bases
df_pop = pd.read_csv(f'{caminho}Agregados_por_bairros_basico_BR.csv', sep=';', encoding='latin1') #IBGE_2024
df_ubs = pd.read_csv(f'{caminho}Unidades_Básicas_de_Saúde_UBS.csv', encoding='utf-8')
df_hosp = pd.read_csv(f'{caminho}Hospitais.csv', encoding='utf-8')
df_poli = pd.read_csv(f'{caminho}Policlínicas.csv', encoding='utf-8')
# df_aten = pd.read_csv(f'{caminho}Area_de_Atendimento_UBS.csv', encoding='utf-8')

print("Arquivos ok.\n")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.0/338.0 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 26.7 MB/s eta 0:00:00
Carregando arquivos...
Arquivos ok.



1. Os arquivos CSV geralmente contêm colunas com códigos de identificação (GEOCODIGODOB).<br><br>
     Nome_Bairro: Nome oficial do bairro.<br>
   - [x] V001: População Total.<br>


###### =========================
###### POPULAÇÃO (IBGE)
###### =========================

In [ ]:
print("Filtrando dados de Niterói...")
pop = df_pop[df_pop['NM_MUN'] == 'Niterói'].copy()

# limpeza de texto
pop['NM_BAIRRO'] = pop['NM_BAIRRO'].str.strip().str.upper()

# tratando população (vem com ponto de milhar)
pop['v0001'] = (
    pop['v0001']
    .astype(str)
    .str.replace('.', '', regex=False)
)

pop['v0001'] = pd.to_numeric(pop['v0001'], errors='coerce')

# renomeando pra algo legível
pop.rename(columns={'v0001': 'POPULACAO'}, inplace=True)

print(f"{len(pop)} bairros encontrados.\n")
display(pop[['NM_BAIRRO', 'POPULACAO']].head())


Filtrando dados de Niterói...
52 bairros encontrados.



,NM_BAIRRO,POPULACAO
8642,BADU,7055
8643,BALDEADOR,4335
8644,BARRETO,18699
8645,BOA VIAGEM,1944
8646,CACHOEIRA,2998


###### =========================
###### BASES DE SAÚDE
###### =========================


In [ ]:
print("Padronizando nomes de bairros...")

def limpar_bairro(serie):
    return (
        serie
        .str.normalize('NFKD')
        .str.encode('ascii', errors='ignore')
        .str.decode('utf-8')
        .str.strip()
        .str.upper()
    )

df_ubs['NM_BAIRRO'] = limpar_bairro(df_ubs['bairro'])
df_hosp['NM_BAIRRO'] = limpar_bairro(df_hosp['tx_bairro'])
df_poli['NM_BAIRRO'] = limpar_bairro(df_poli['bairro'])


# contagem por bairro
qtd_ubs = df_ubs['NM_BAIRRO'].value_counts().reset_index(name='QTD_UBS')
qtd_hosp = df_hosp['NM_BAIRRO'].value_counts().reset_index(name='QTD_HOSP')
qtd_poli = df_poli['NM_BAIRRO'].value_counts().reset_index(name='QTD_POLI')


# juntando tudo
infra = qtd_ubs.merge(qtd_hosp, on='NM_BAIRRO', how='outer')
infra = infra.merge(qtd_poli, on='NM_BAIRRO', how='outer')

# tratando NaN
infra = infra.fillna(0).infer_objects(copy=False)

# criando colunas auxiliares
infra['QTD_BASICA'] = infra['QTD_UBS'] + infra['QTD_POLI']
infra['TOTAL_GERAL'] = infra['QTD_BASICA'] + infra['QTD_HOSP']

print("Infraestrutura consolidada.\n")
display(infra.sort_values('TOTAL_GERAL', ascending=False))


Padronizando nomes de bairros...
Infraestrutura consolidada.



,NM_BAIRRO,QTD_UBS,QTD_HOSP,QTD_POLI,QTD_BASICA,TOTAL_GERAL
1,BARRETO,1.0,4.0,1.0,2.0,6.0
5,FONSECA,1.0,4.0,1.0,2.0,6.0
2,CENTRO,2.0,3.0,1.0,3.0,6.0
13,PIRATININGA,1.0,1.0,1.0,2.0,3.0
16,VITAL BRAZIL,1.0,0.0,1.0,2.0,2.0
11,MARAVISTA,1.0,0.0,1.0,2.0,2.0
15,SANTA ROSA,0.0,2.0,0.0,0.0,2.0
10,LARGO DA BATALHA,1.0,0.0,1.0,2.0,2.0
4,ENGENHOCA,1.0,0.0,1.0,2.0,2.0
0,BAIRRO DE FATIMA,0.0,1.0,0.0,0.0,1.0


##### =========================
##### INTEGRAÇÃO FINAL
##### =========================

In [ ]:
print("Integrando bases...")

# garantir padrão igual
pop['NM_BAIRRO'] = limpar_bairro(pop['NM_BAIRRO'])

final = pop.merge(infra, on='NM_BAIRRO', how='left')
final = final.fillna(0).infer_objects(copy=False)

# habitantes por unidade
final['HAB_POR_UBS_POLI'] = final['POPULACAO'] / final['QTD_BASICA']

# substitui infinito por NaN
final['HAB_POR_UBS_POLI'] = final['HAB_POR_UBS_POLI'].replace([np.inf, -np.inf], np.nan)

# agora sim arredonda e converte
final['HAB_POR_UBS_POLI'] = final['HAB_POR_UBS_POLI'].round().astype('Int64')

# estatísticas só onde existe atendimento
atendidos = final[final['QTD_BASICA'] > 0]

media = atendidos['HAB_POR_UBS_POLI'].mean()
desvio = atendidos['HAB_POR_UBS_POLI'].std()

# z-score
final['Z_SCORE_DEMANDA'] = np.where(
    final['QTD_BASICA'] > 0,
    ((final['HAB_POR_UBS_POLI'] - media) / desvio).round(2),
    np.nan
)

# evitar alertas chatos
pd.set_option('future.no_silent_downcasting', True)

# só pra exibir bonito
vis = final.fillna(0)

print(f"\nProcessamento finalizado ({len(final)} bairros).")
print(f"Média da cidade: {media:.0f} hab/unidade\n")

display(
    vis[['NM_BAIRRO', 'POPULACAO', 'QTD_BASICA', 'HAB_POR_UBS_POLI', 'Z_SCORE_DEMANDA']]
    .sort_values('QTD_BASICA', ascending=False)
)

Integrando bases...

Processamento finalizado (52 bairros).
Média da cidade: 8435 hab/unidade



/tmp/ipykernel_50127/2637639131.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  final = final.fillna(0).infer_objects(copy=False)


,NM_BAIRRO,POPULACAO,QTD_BASICA,HAB_POR_UBS_POLI,Z_SCORE_DEMANDA
9,CENTRO,17938,3.0,5979,-0.42
2,BARRETO,18699,2.0,9350,0.15
12,ENGENHOCA,16700,2.0,8350,-0.01
15,FONSECA,46317,2.0,23158,2.49
32,PIRATININGA,18492,2.0,9246,0.14
49,MARAVISTA,10584,2.0,5292,-0.53
25,LARGO DA BATALHA,9637,2.0,4818,-0.61
29,MORRO DO ESTADO,3111,1.0,3111,-0.90
35,SANTA BARBARA,6609,1.0,6609,-0.31
3,BOA VIAGEM,1944,0.0,0,0.00


In [ ]:
import pandas as pd
from scipy import stats

# 1. Padronização e Limpeza (Tratamento de Veracidade - Etapa 3)
# Ajustando as colunas de bairro de cada base (UBS, Hospitais, Policlínicas)
df_ubs['BAIRRO_LIMPO'] = df_ubs['bairro'].str.upper().str.strip()
df_hosp['BAIRRO_LIMPO'] = df_hosp['tx_bairro'].str.upper().str.strip()
df_poli['BAIRRO_LIMPO'] = df_poli['bairro'].str.upper().str.strip()

# 2. Identificar ONDE está cada unidade (Localização por Bairro)
# Criamos uma lista com os nomes das unidades em cada bairro
localizacao_detalhada = df_hosp.groupby('BAIRRO_LIMPO')['tx_nome'].apply(lambda x: ", ".join(x)).reset_index(name='HOSPITAIS')
localizacao_ubs = df_ubs.groupby('BAIRRO_LIMPO')['nome'].apply(lambda x: ", ".join(x)).reset_index(name='UBSs')

# 3. Contagem para Análise Estatística (Etapa 4 e 5)
ubs_count = df_ubs['BAIRRO_LIMPO'].value_counts().reset_index(name='QTD_UBS')
hosp_count = df_hosp['BAIRRO_LIMPO'].value_counts().reset_index(name='QTD_HOSP')
poli_count = df_poli['BAIRRO_LIMPO'].value_counts().reset_index(name='QTD_POLI')

# Fixando o Merge: Usamos 'BAIRRO_LIMPO' em vez de 'index'
df_mapa_saude = pd.merge(ubs_count, hosp_count, on='BAIRRO_LIMPO', how='outer')
df_mapa_saude = pd.merge(df_mapa_saude, poli_count, on='BAIRRO_LIMPO', how='outer').fillna(0)

# 4. Cálculo de Indicadores Totais
df_mapa_saude['Total_Equipamentos'] = df_mapa_saude['QTD_UBS'] + df_mapa_saude['QTD_HOSP'] + df_mapa_saude['QTD_POLI']

# 5. Análise Estatística Descritiva (Média e Desvio Padrão)
media = df_mapa_saude['Total_Equipamentos'].mean()
desvio_padrao = df_mapa_saude['Total_Equipamentos'].std()

# 6. Identificação de Outliers via Z-Score (Etapa 7)
df_mapa_saude['Z_Score'] = stats.zscore(df_mapa_saude['Total_Equipamentos'])

print(f"--- Estatísticas de Niterói ---")
print(f"Média de equipamentos: {media:.2f}")
print(f"Desvio Padrão (Dispersão): {desvio_padrao:.2f}\n")

# Exibindo os bairros com mais concentração (Outliers)
outliers = df_mapa_saude[df_mapa_saude['Z_Score'] > 1.5].sort_values('Total_Equipamentos', ascending=False)
print("Bairros Identificados como Polos de Saúde (Outliers):")
display(outliers[['BAIRRO_LIMPO', 'Total_Equipamentos', 'Z_Score']])

--- Estatísticas de Niterói ---
Média de equipamentos: 2.29
Desvio Padrão (Dispersão): 1.86

Bairros Identificados como Polos de Saúde (Outliers):


,BAIRRO_LIMPO,Total_Equipamentos,Z_Score
1,BARRETO,6.0,2.050475
2,CENTRO,6.0,2.050475
5,FONSECA,6.0,2.050475


In [ ]:
# Exemplo de como classificar no Pandas
def categorizar_gestao(nome):
    nome = nome.upper()
    if 'MUNICIPAL' in nome or 'ESTADUAL' in nome or 'FEDERAL' in nome or 'UNIVERSITÁRIO' in nome:
        return 'PÚBLICO'
    else:
        return 'PRIVADO/FILANTRÓPICO'

df_hosp['CATEGORIA'] = df_hosp['tx_nome'].apply(categorizar_gestao)
# 1. Aplica a função para criar a nova coluna 'CATEGORIA'
df_hosp['CATEGORIA'] = df_hosp['tx_nome'].apply(categorizar_gestao)

# 2. Imprime as primeiras 5 linhas mostrando o Nome e a Categoria
print("--- Amostra da Classificação ---")
print(df_hosp[['tx_nome', 'CATEGORIA']])

--- Amostra da Classificação ---
                                              tx_nome             CATEGORIA
0                      Complexo Hospitalar de Niterói  PRIVADO/FILANTRÓPICO
1                        Hospital de Clínicas do Ingá  PRIVADO/FILANTRÓPICO
2                Hospital Universitário Antônio Pedro               PÚBLICO
3                  Hospital Municipal Carlos Tortelly               PÚBLICO
4                   Hospital Psiquiátrico de Jurujuba  PRIVADO/FILANTRÓPICO
5                        Hospital de Clinicas Alameda  PRIVADO/FILANTRÓPICO
6                           Hospital de Olhos Niteroi  PRIVADO/FILANTRÓPICO
7                                     Hospital Icarai  PRIVADO/FILANTRÓPICO
8                       Hospital Getulio Vargas Filho  PRIVADO/FILANTRÓPICO
9                         Hospital da Polícia Militar  PRIVADO/FILANTRÓPICO
10                    Hospital de Olhos Santa Beatriz  PRIVADO/FILANTRÓPICO
11                 Hospital de Clínicas São Sebastião  

In [ ]:
try:
# --- TAREFA 2 E 3: LIMPEZA E VERACIDADE ---
  def localizar_coluna_bairro(df):
  # Procura por variações de nome de coluna que contenham 'BAIRRO'
    for col in df.columns:
      if 'BAIRRO' in col.upper():
        return col
    return None

# Normalizando e consolidando a contagem
  counts = []

  for nome, df_current in [('UBS', df_ubs), ('HOSPITAL', df_hosp), ('POLI', df_poli)]:
    col_bairro = localizar_coluna_bairro(df_current)
  if col_bairro:
    df_current[col_bairro] = df_current[col_bairro].str.upper().str.strip()
  count = df_current[col_bairro].value_counts().reset_index()
  count.columns = ['BAIRRO_LIMPO', f'QTD_{nome}']
  counts.append(count)

# Cruzamento de dados (Merge)
  df_indicadores = counts[0]
  for c in counts[1:]:
    df_indicadores = pd.merge(df_indicadores, c, on='BAIRRO_LIMPO', how='outer')

  df_indicadores = df_indicadores.fillna(0)
  df_indicadores['TOTAL_EQUIPAMENTOS'] = df_indicadores.iloc[:, 1:].sum(axis=1)

  # --- TAREFA 4: ANÁLISE ESTATÍSTICA (MÉDIA, DESVIO PADRÃO E OUTLIERS) ---
  print("--- 2. Processando Estatística Descritiva ---")

  # Cálculo das métricas solicitadas na estrutura
  media = df_indicadores['TOTAL_EQUIPAMENTOS'].mean()
  mediana = df_indicadores['TOTAL_EQUIPAMENTOS'].median()
  desvio_padrao = df_indicadores['TOTAL_EQUIPAMENTOS'].std()

  # Identificação de Outliers (Z-Score) para detectar anomalias
  df_indicadores['z_score'] = stats.zscore(df_indicadores['TOTAL_EQUIPAMENTOS'])

  print(f"Média de Equipamentos por Bairro: {media:.2f}")
  print(f"Desvio Padrão (Dispersão): {desvio_padrao:.2f}")

  # Relatório de Outliers (Bairros com oferta atípica)
  outliers = df_indicadores[abs(df_indicadores['z_score']) > 1.5]
  print("\n--- Relatório de Outliers (Anomalias de Demanda) ---")
  display(outliers[['BAIRRO_LIMPO', 'TOTAL_EQUIPAMENTOS', 'z_score']])

  # --- TAREFA: CARGA E VISUALIZAÇÃO (LOAD) ---
  print("--- 3. Gerando Visualização Final (Impacto Social) ---")

  # Mapa de Bairros via Geobr
  malha = geobr.read_neighborhood(year=2010)
  malha = malha[malha['code_muni'] == 3303302].copy()
  malha['name_neighborhood'] = malha['name_neighborhood'].str.upper().str.strip()

  # Unindo dados ao mapa
  mapa_final = malha.merge(df_indicadores, left_on='name_neighborhood', right_on='BAIRRO_LIMPO', how='left').fillna(0)

  # Criação do Mapa Folium
  m = folium.Map(location=[-22.8833, -43.1036], zoom_start=13, tiles='cartodbpositron')

  def style_function(feature):
    z = feature['properties']['z_score']
    if z > 1.5: color = '#9b59b6' # Roxo: Outlier (Alta concentração)
    elif z < -1: color = '#e74c3c' # Vermelho: Vazio Assistencial
    elif feature['properties']['TOTAL_EQUIPAMENTOS'] > 0: color = '#2ecc71' # Verde: Atendimento Normal
    else: color = '#bdc3c7' # Cinza: Sem dados
    return {'fillColor': color, 'color': 'white', 'weight': 1, 'fillOpacity': 0.7}

    folium.GeoJson(mapa_final, style_function=style_function,
  tooltip=folium.GeoJsonTooltip(fields=['name_neighborhood', 'TOTAL_EQUIPAMENTOS', 'z_score'],
  aliases=['Bairro:', 'Equipamentos:', 'Z-Score:'])
  ).add_to(m)

  display(m)
  print("Projeto Concluído: Saneamento, Estatística e Carga finalizados.")
except Exception as e:
  print(f"Erro na conclusão: {e}")

--- 2. Processando Estatística Descritiva ---
Média de Equipamentos por Bairro: 1.00
Desvio Padrão (Dispersão): 0.00

--- Relatório de Outliers (Anomalias de Demanda) ---


/tmp/ipykernel_50127/2914212035.py:38: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  df_indicadores['z_score'] = stats.zscore(df_indicadores['TOTAL_EQUIPAMENTOS'])


,BAIRRO_LIMPO,TOTAL_EQUIPAMENTOS,z_score


--- 3. Gerando Visualização Final (Impacto Social) ---


Projeto Concluído: Saneamento, Estatística e Carga finalizados.


In [ ]:
from pysus.ftp.databases.cnes import CNES
import pandas as pd

def baixar_dados_cnes_estavel(ano, mes):
    # Instancia a classe do CNES
    cnes = CNES()
    codigo_niteroi = '330330'

    try:
        print(f"--- 1. Coleta e Saneamento: {mes}/{ano} ---")

        # O método download no CNES geralmente espera (UF, ANO, MES, GRUPO)
        # 'ST' = Estabelecimentos | 'LT' = Leitos

        print("Baixando Estabelecimentos (ST)...")
        df_st = cnes.download('RJ', ano, mes, group='ST').to_dataframe()
        df_st_nit = df_st[df_st['CODUFMUN'] == codigo_niteroi].copy()

        print("Baixando Leitos (LT)...")
        df_lt = cnes.download('RJ', ano, mes, group='LT').to_dataframe()
        df_lt_nit = df_lt[df_lt['CODUFMUN'] == codigo_niteroi].copy()

        # Caminho do seu Drive
        caminho = '/content/drive/MyDrive/Colab Notebooks/DATA/'

        df_st_nit.to_csv(f'{caminho}CNES_ST_Niteroi.csv', index=False)
        df_lt_nit.to_csv(f'{caminho}CNES_LT_Niteroi.csv', index=False)

        print("✅ Dados de Niterói salvos com sucesso!")
        return df_st_nit, df_lt_nit

    except Exception as e:
        print(f"❌ Erro na coleta: {e}")
        # Dica: Se o erro persistir, tente remover o argumento group='ST'
        # e usar apenas cnes.download('RJ', ano, mes)
        return None, None

# Tentar a execução
df_st, df_lt = baixar_dados_cnes_estavel(2024, 1)

--- 1. Coleta e Saneamento: 1/2024 ---
Baixando Estabelecimentos (ST)...
❌ Erro na coleta: Database.download() got an unexpected keyword argument 'group'
